# [9665] GloVe 1

In [1]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 04/01/25 17:07:35


### Import libraries

In [2]:
import numpy as np
import nltk
import time
from nltk.corpus import gutenberg
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [3]:
# Download required NLTK datasets
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package gutenberg to /Users/vj/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /Users/vj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/vj/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/vj/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

### Load data

In [4]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

In [5]:
# Create function to load and preprocess Gutenberg corpus
def load_gutenberg_corpus():
    print("Loading and preprocessing Gutenberg corpus...")
    sentences = gutenberg.sents()
    corpus = [[lemmatizer.lemmatize(word.lower()) for word in sentence if word.isalpha()] for sentence in sentences]
    return corpus

In [6]:
%%time

# Load Gutenberg corpus
corpus = load_gutenberg_corpus()
print(f"# sentences in corpus: {len(corpus)}")

Loading and preprocessing Gutenberg corpus...
# sentences in corpus: 98552
CPU times: user 13 s, sys: 485 ms, total: 13.5 s
Wall time: 13.8 s


### Generate co-occurrence matrix

In [7]:
# Create function to build Co-occurrence Matrix
def build_cooccurrence_matrix(corpus, vocab_size=5000, window_size=5):
    print("Building co-occurrence matrix...")
    word_counts = {}
    
    # Count word frequencies
    for sentence in corpus:
        for word in sentence:
            if word in word_counts:
                word_counts[word] += 1
            else:
                word_counts[word] = 1
    
    # Select the top vocab_size words based on frequency
    sorted_vocab = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:vocab_size]
    vocab = {word: i for i, (word, _) in enumerate(sorted_vocab)}
    
    # Initialize co-occurrence matrix
    cooccurrence = {}
    for sentence in corpus:
        for i, word in enumerate(sentence):
            if word in vocab:
                if vocab[word] not in cooccurrence:
                    cooccurrence[vocab[word]] = {}
                
                # Count word co-occurrences within the window size
                for j in range(max(i - window_size, 0), min(i + window_size + 1, len(sentence))):
                    if i != j and sentence[j] in vocab:
                        index_j = vocab[sentence[j]]
                        if index_j in cooccurrence[vocab[word]]:
                            cooccurrence[vocab[word]][index_j] += 1
                        else:
                            cooccurrence[vocab[word]][index_j] = 1
    
    return vocab, cooccurrence

In [8]:
%%time

# Build co-occurrence matrix
vocab, cooccurrence = build_cooccurrence_matrix(corpus)

Building co-occurrence matrix...
CPU times: user 9.97 s, sys: 161 ms, total: 10.1 s
Wall time: 10.5 s


In [9]:
# Create a mapping from words to their indices (assuming vocab is available)
word_to_index = {word: i for i, word in enumerate(vocab)}

### Train GloVe model using a simple implementation

In [10]:
# Create function to train GloVe embedding
def train_glove(cooccurrence, vocab_size, vector_size=50, alpha=0.75, x_max=100,
                epochs=5, learning_rate=0.01, clip_value=5.0):
    print("Training GloVe model...")
    
    # Initialize word vectors and biases with small values
    W = np.random.rand(vocab_size, vector_size) * 0.01
    W_tilde = np.random.rand(vocab_size, vector_size) * 0.01
    biases = np.random.rand(vocab_size) * 0.01
    biases_tilde = np.random.rand(vocab_size) * 0.01
    
    for epoch in range(epochs):
        start_time = time.time()
        total_loss = 0
        for i in cooccurrence:
            for j in cooccurrence[i]:
                X_ij = cooccurrence[i][j]
                log_X_ij = np.log(X_ij + 1)  # Use log to avoid overflow
                weight = (X_ij / x_max) ** alpha if X_ij < x_max else 1  # Compute weighting function
                diff = np.dot(W[i], W_tilde[j]) + biases[i] + biases_tilde[j] - log_X_ij
                loss = weight * diff ** 2  # Compute loss
                total_loss += loss
                
                grad = 2 * weight * diff  # Compute gradient
                grad = np.clip(grad, -clip_value, clip_value)  # Clip gradients to avoid overflow
                
                # Update parameters
                W[i] -= learning_rate * grad * W_tilde[j]
                W_tilde[j] -= learning_rate * grad * W[i]
                biases[i] -= learning_rate * grad
                biases_tilde[j] -= learning_rate * grad
        
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}, Time: {epoch_time:.2f} seconds")
    
    return W

In [11]:
%%time

# Generate GloVe embeddings
#embeddings = train_glove(cooccurrence, len(vocab))
W = train_glove(cooccurrence, len(vocab))

Training GloVe model...
Epoch 0, Loss: 154811.8380, Time: 50.03 seconds
Epoch 1, Loss: 66919.5264, Time: 48.57 seconds
Epoch 2, Loss: 61883.6207, Time: 45.08 seconds
Epoch 3, Loss: 60069.0896, Time: 44.26 seconds
Epoch 4, Loss: 58960.7277, Time: 43.22 seconds
CPU times: user 3min 46s, sys: 2.24 s, total: 3min 48s
Wall time: 3min 51s


### Use trained GloVe model

#### Create functions to ask questions of GloVe model

In [12]:
# Create the embeddings dictionary (mapping words to vectors)
def get_embeddings(W, vocab):
    embeddings = {}
    for word, index in word_to_index.items():
        embeddings[word] = W[index]
    return embeddings

# Compute cosine similarity between two word vectors
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Find the most similar words based on cosine similarity
def most_similar_words(word, topn=5, embeddings=None):
    if word not in embeddings:
        return f"'{word}' not found in vocabulary."
    word_vec = embeddings[word]
    similarities = {
        other_word: cosine_similarity(word_vec, other_vec)
        for other_word, other_vec in embeddings.items()
        if other_word != word
    }
    return sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:topn]

# Find the word that does not belong based on cosine similarity
def find_odd_one_out(words, embeddings):
    similarities = {}
    
    # Calculate average similarity for each word in the list
    for word in words:
        if word not in embeddings:
            return f"'{word}' not found in vocabulary."
        word_vec = embeddings[word]
        avg_similarity = 0
        for other_word in words:
            if word != other_word and other_word in embeddings:
                avg_similarity += cosine_similarity(word_vec, embeddings[other_word])
        
        similarities[word] = avg_similarity / (len(words) - 1)  # Average similarity to other words
    
    # Find the word with the smallest average similarity (the odd one out)
    odd_one_out = min(similarities, key=similarities.get)
    return odd_one_out

In [13]:
# Get the embeddings dictionary
embeddings = get_embeddings(W, vocab)

#### Ask questions of GloVe model

In [14]:
# Example: Find most similar words to 'king'
print("Most similar words to 'king':", most_similar_words("king", topn=5, embeddings=embeddings))

Most similar words to 'king': [('son', 0.9969966602081323), ('prince', 0.9940529260349302), ('judah', 0.9935502442140689), ('priest', 0.9933884081434682), ('elder', 0.992570861070135)]


In [15]:
# Example: Find most similar words to 'queen'
print("Most similar words to 'queen':", most_similar_words("queen", topn=5, embeddings=embeddings))

Most similar words to 'queen': [('river', 0.9925550275484172), ('hill', 0.9918978865676826), ('light', 0.991771465150658), ('levite', 0.9916892483885952), ('midst', 0.9915417258048286)]


In [16]:
# Example: Find most similar words to 'nonexistentword'
print("Most similar words to 'nonexistentword':", most_similar_words("nonexistentword", topn=5, embeddings=embeddings))

Most similar words to 'nonexistentword': 'nonexistentword' not found in vocabulary.


In [17]:
words_1 = ["breakfast", "supper", "dinner", "car"]
words_2 = ["king", "apple", "queen", "man"]
words_3 = ["dog", "cat", "fish", "car"]

In [18]:
# Example: Find odd one out in the context of meal-related words
print("Odd one out from ['breakfast', 'supper', 'dinner', 'car']:", find_odd_one_out(words_1, embeddings))

Odd one out from ['breakfast', 'supper', 'dinner', 'car']: supper


In [19]:
# Example: Find odd one out in the context of royalty-related words
print("Odd one out from ['king', 'apple', 'queen', 'man']:", find_odd_one_out(words_2, embeddings))

Odd one out from ['king', 'apple', 'queen', 'man']: apple


In [20]:
# Example: Find odd one out in the context of animals
print("Odd one out from ['dog', 'cat', 'fish', 'car']:", find_odd_one_out(words_3, embeddings))

Odd one out from ['dog', 'cat', 'fish', 'car']: car
